# Tesis de maestría en Ciencia de Datos
## Proceso completo reproducible: calidad, armonización y fusión multifuente de datos de dengue

**Versión de trabajo:** 5 de noviembre de 2026 · **Entorno:** Google Colab · **Unidad:** registro geográfico–periodo de brote.

Este es **un solo cuaderno integral**. Reúne lo que se ha realizado hasta ahora, desde la inspección de las bases históricas y la base maestra hasta el diagnóstico de calidad, la fusión temprana y tardía y la elaboración de tablas para la tesis.

**Archivo requerido:** `fusion_base_maestra_4_brotes_ACTUALIZADA.csv`. Es la base maestra construida anteriormente a partir de las cuatro bases corregidas; contiene 68,356 observaciones y 67 columnas. **Archivos opcionales:** los cuatro archivos integrados para comprobar sus filas y columnas y `fusion_resultados_exploratorios.csv` para contrastar la tabla del primer avance.

**Alcance:** se reproduce el procedimiento exploratorio con los tres grupos originales, se ejecuta un nuevo experimento **con las coberturas históricas** y se generan diagnósticos y reportes. No se afirma que la evaluación nueva ya se haya realizado: los resultados se calculan al ejecutar este cuaderno.

**Precaución:** el cuarto brote se reserva. Las cifras de validación no son resultados de evaluación final, y los registros aún requieren validar las correspondencias de algunas geometrías.

## 0. Guía de ejecución
1. Ejecutar las celdas **en orden**, desde arriba hasta abajo.
2. Subir la base maestra actualizada cuando aparezca la ventana.
3. Dejar `EJECUTAR_MODELOS = True` para ejecutar ambos enfoques de fusión.
4. Revisar el resumen de calidad y los resultados de validación antes de redactar conclusiones.
5. Al final se genera un **ZIP con tablas y gráficos**, además de CSV de predicciones y métricas.

No se eliminan ni sobrescriben las bases originales. Las transformaciones se realizan en memoria.

In [ ]:
# 0. Bibliotecas habituales de Google Colab.
# Solo se ejecutará la instalación si alguna dependencia falta.
import sys, subprocess, importlib.util
for paquete in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn']:
    modulo = 'sklearn' if paquete == 'scikit-learn' else paquete
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])

import re, unicodedata, zipfile, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

# Parámetros globales: cambiar solo si se requiere otro experimento.
SEMILLA = 123
EJECUTAR_MODELOS = True
CARPETA = Path('salidas_fusion_multifuente')
CARPETA.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('Entorno listo. La semilla aleatoria es', SEMILLA)

## 1. Carga y validación de la base maestra
La integración de las bases 1–4 se efectuó previamente, preservando sus registros, el periodo, una clave técnica de fila, la variable `casos_dengue` y las coberturas históricas. **Este cuaderno lee la versión corregida**, no las versiones antiguas del segundo brote (2013–2014).

Si se trabaja en Colab, se solicita subir el CSV. Si se ejecuta en una computadora donde el archivo ya existe, se utiliza esa copia.

In [ ]:
# 1. Seleccionar la base maestra.
NOMBRE_MAESTRA = 'fusion_base_maestra_4_brotes_ACTUALIZADA.csv'
rutas = [Path(NOMBRE_MAESTRA), Path('/content') / NOMBRE_MAESTRA, Path('/mnt/data') / NOMBRE_MAESTRA]
archivo = next((q for q in rutas if q.exists()), None)
if archivo is None:
    from google.colab import files
    subidos = files.upload()
    if NOMBRE_MAESTRA in subidos:
        archivo = Path(NOMBRE_MAESTRA)
    else:
        candidatos = [Path(k) for k in subidos if k.lower().endswith('.csv')]
        if len(candidatos) != 1:
            raise FileNotFoundError('Sube la base maestra actualizada e intenta nuevamente.')
        archivo = candidatos[0]

# Se conservan las cadenas originales y las columnas numéricas tal como vienen.
df = pd.read_csv(archivo, low_memory=False, encoding='utf-8-sig')
print('Archivo cargado:', archivo)
print('Dimensiones (filas, columnas):', df.shape)
display(df.head(3).drop(columns=[c for c in df if c.startswith('pct_')], errors='ignore'))

In [ ]:
# 2. Comprobaciones imprescindibles de estructura e identidad.
obligatorias = ['id_fila_original','brote','periodo','ciudad','casos_dengue',
                'dosel_pct_media_periodo','urbano_pct_media_periodo']
no_existen = [c for c in obligatorias if c not in df.columns]
if no_existen:
    raise ValueError(f'Faltan columnas obligatorias: {no_existen}')

# El identificador de fila es técnico: NO equivale a una clave oficial AGEB.
print('Observaciones por brote y periodo:')
display(df.groupby(['brote','periodo'], dropna=False).size().rename('n').reset_index())
print('Duplicados de (brote, id_fila_original):',
      int(df.duplicated(['brote','id_fila_original']).sum()))
print('Ciudades diferentes (sin normalización):', df['ciudad'].nunique(dropna=True))
print('Número de registros con conteo ausente:', int(df['casos_dengue'].isna().sum()))
# Prohibimos continuar si falla la clave técnica en la base maestra.
if df.duplicated(['brote','id_fila_original']).any():
    raise ValueError('Hay claves de fila duplicadas dentro de un brote; verificar antes del modelado.')

### 1.1. Auditoría opcional de las cuatro bases de origen
Cuando estén disponibles en la sesión, esta sección comprueba que los cuatro archivos integrados tienen el mismo número de filas y que los años de cobertura son los correctos. **No los une de nuevo por posición**, porque hubo polígonos ausentes y cambios de orden en descargas anteriores. La base maestra previamente auditada es la fuente de trabajo.

In [ ]:
# 3. Auditoría opcional: no interrumpe la ejecución si faltan los CSV de origen.
archivos_origen = {
  1: 'brote1_integrado_coberturas_2008_2009.csv',
  2: 'brote2_integrado_coberturas_2012_2013_CORREGIDO.csv',
  3: 'brote3_integrado_coberturas_2018_2020.csv',
  4: 'brote4_integrado_coberturas_2023_2025.csv',
}
anios = {1:[2008,2009], 2:[2012,2013], 3:[2018,2019,2020], 4:[2023,2024,2025]}
filas_auditoria = []
for brote, nombre in archivos_origen.items():
    ruta = next((p for p in [Path(nombre), Path('/content')/nombre, Path('/mnt/data')/nombre] if p.exists()), None)
    if ruta is None:
        filas_auditoria.append({'brote':brote,'archivo':nombre,'estado':'No cargado (opcional)'})
        continue
    # Solo cabecera y número de filas; evitar cargar nuevamente geometrías enormes.
    origen = pd.read_csv(ruta, low_memory=False, usecols=lambda c: not (c in ['geometry','geometry_wkt']))
    requeridas = [f'pct_{tipo}_{anio}' for tipo in ['dosel','urbano'] for anio in anios[brote]]
    faltan = [c for c in requeridas if c not in origen]
    filas_auditoria.append({'brote':brote,'archivo':nombre,'n':len(origen),
                            'faltan_columnas':', '.join(faltan),
                            'estado':'revisar' if faltan or len(origen)!=17089 else 'conforme'})
res_origen = pd.DataFrame(filas_auditoria)
display(res_origen)
res_origen.to_csv(CARPETA/'auditoria_archivos_origen.csv',index=False,encoding='utf-8-sig')

## 2. Procedencia, correspondencia y calidad de datos
Las coberturas antiguas `tree`/`urban` provenían de **Copernicus PROBA-V**; las variables históricas nuevas de dosel y superficie construida se obtuvieron con **Hansen** y **GHSL**, respectivamente. En la base maestra se conservan los valores anuales, el primer año y el promedio del periodo.

**Dos precauciones:** (a) la geometría y su correspondencia no se validan simplemente porque haya 17,089 filas por brote; (b) la cobertura de un año posterior al comienzo del brote no sería una variable admisible para pronosticar todo el periodo desde su inicio.

In [ ]:
# 4. Diagnóstico por brote, sin modificar los datos.
# Se conservan los estados heredados del proceso de correspondencia de polígonos.
columnas_revision = ['estado_correspondencia_cobertura','cobertura_correspondencia_exacta',
                     'cobertura_anios_disponibles_dosel','cobertura_anios_disponibles_urbano',
                     'alerta_variables_posteriores_inicio']
for c in columnas_revision:
    if c in df:
        print(f'\nDistribución de {c}:')
        display(df.groupby(['periodo',c], dropna=False).size().rename('n').reset_index().head(40))

# La coincidencia declarada es evidencia de control, NO reemplaza una comprobación geométrica.
alertas_poligonos = df.loc[df['estado_correspondencia_cobertura'].ne('geometria_exacta'),
                            [c for c in ['brote','periodo','ciudad','id_fila_original',
                                          'estado_correspondencia_cobertura','id_poligono_cobertura'] if c in df]].copy()
print('Filas con correspondencia no declarada como exacta:',len(alertas_poligonos))
alertas_poligonos.to_csv(CARPETA/'alertas_correspondencia_poligonos.csv',index=False,encoding='utf-8-sig')

### 2.1. Faltantes estructurales frente a faltantes reales
Una fila de 2008–2009 no debería tener valores de `pct_dosel_2025`. Esos vacíos son **estructurales**: no deben sumarse a los faltantes que realmente afectan el análisis. La revisión principal se hace con columnas comunes a los cuatro periodos.

In [ ]:
# 5. Tabla de valores faltantes en variables comunes.
comunes = [c for c in ['casos_dengue','bio1','bio12','ia','ice','im','irs','hdi',
                       'temp_sui','dhi','poblacion_inicio_periodo',
                       'poblacion_media_disponible','hfp_inicio_periodo',
                       'hfp_media_disponible','dosel_pct_inicio_periodo',
                       'urbano_pct_inicio_periodo','dosel_pct_media_periodo',
                       'urbano_pct_media_periodo'] if c in df.columns]
faltantes = (df.groupby('periodo',sort=False)[comunes]
             .apply(lambda x: x.isna().sum())
             .T)
faltantes_pct = (df.groupby('periodo',sort=False)[comunes]
                 .apply(lambda x: x.isna().mean()*100).T)
print('Faltantes absolutos:')
display(faltantes)
print('Faltantes porcentuales:')
display(faltantes_pct.round(2))
faltantes.to_csv(CARPETA/'faltantes_comunes_conteos.csv',encoding='utf-8-sig')
faltantes_pct.to_csv(CARPETA/'faltantes_comunes_porcentajes.csv',encoding='utf-8-sig')
plt.figure(figsize=(10,7))
sns.heatmap(faltantes_pct,annot=True,fmt='.1f',cmap='YlOrRd',vmin=0)
plt.title('Porcentaje de faltantes por variable y periodo')
plt.xlabel('Periodo');plt.ylabel('Variable');plt.tight_layout()
plt.savefig(CARPETA/'fig_faltantes_por_periodo.png',dpi=170);plt.show()

In [ ]:
# 6. Validez de dominios y detección de valores extremos no equivale a depurarlos.
y = pd.to_numeric(df['casos_dengue'], errors='coerce')
print('Conteos faltantes:',int(y.isna().sum()))
print('Conteos negativos:',int((y < 0).sum()))
print('Conteos no enteros:',int(((y % 1 != 0) & y.notna()).sum()))
cols_pct = [c for c in df if c.startswith('pct_dosel_') or c.startswith('pct_urbano_')]
cols_pct += [c for c in ['dosel_pct_inicio_periodo','urbano_pct_inicio_periodo',
                        'dosel_pct_media_periodo','urbano_pct_media_periodo'] if c in df]
dominio = []
for c in cols_pct:
    v = pd.to_numeric(df[c],errors='coerce')
    dominio.append({'variable':c,'n_disponibles':int(v.notna().sum()),
                    'fuera_0_100':int(((v<0)|(v>100)).sum()),
                    'min':v.min(),'max':v.max()})
tabla_dominio = pd.DataFrame(dominio)
display(tabla_dominio)
tabla_dominio.to_csv(CARPETA/'revision_dominio_coberturas.csv',index=False,encoding='utf-8-sig')
if (y<0).any() or (((y%1!=0)&y.notna()).any()):
    raise ValueError('La variable objetivo tiene valores fuera del dominio de conteos.')

## 3. Análisis exploratorio de los brotes
Se describen la frecuencia de ceros, la dispersión y los extremos del conteo. Los periodos duran distinto número de años, por lo cual **sus conteos acumulados no son automáticamente tasas comparables**. La comparación se presenta como descripción, no como evidencia de cambio de riesgo.

In [ ]:
# 7. Distribución de casos por periodo.
def resumen_casos(g):
    s = pd.to_numeric(g['casos_dengue'], errors='coerce')
    return pd.Series({'n':len(s), 'total_casos':s.sum(), 'media':s.mean(),
                      'mediana':s.median(), 'desv_est':s.std(),
                      'maximo':s.max(), 'pct_ceros':100*s.eq(0).mean(),
                      'varianza':s.var(), 'razon_var_media':s.var()/s.mean() if s.mean()>0 else np.nan})
tabla_casos = df.groupby('periodo',sort=False).apply(resumen_casos)
display(tabla_casos.round(3))
tabla_casos.to_csv(CARPETA/'resumen_casos_brotes.csv',encoding='utf-8-sig')
fig,ax=plt.subplots(1,2,figsize=(13,4.5))
ax[0].bar(tabla_casos.index.astype(str),tabla_casos['pct_ceros'])
ax[0].set(title='Observaciones con cero casos',ylabel='Porcentaje (%)',xlabel='')
ax[1].bar(tabla_casos.index.astype(str),tabla_casos['media'])
ax[1].set(title='Media de casos observados',ylabel='Casos por registro',xlabel='')
for a in ax: a.tick_params(axis='x',rotation=20)
plt.tight_layout();plt.savefig(CARPETA/'fig_distribucion_casos_brotes.png',dpi=170);plt.show()

In [ ]:
# 8. Histograma restringido visualmente a percentil 99 para mostrar los ceros.
# No se eliminan los extremos de la base ni del entrenamiento.
limite = max(1, float(y.quantile(.99)))
plt.figure(figsize=(9,4.5))
plt.hist(y.dropna().clip(upper=limite),bins=30)
plt.xlabel('Casos de dengue (valores mayores truncados solo en la gráfica)')
plt.ylabel('Frecuencia');plt.title('Distribución exploratoria de casos')
plt.tight_layout();plt.savefig(CARPETA/'fig_histograma_casos.png',dpi=170);plt.show()
print('Límite visual (percentil 99):',round(limite,3))

### 3.1. Coberturas armonizadas
Los porcentajes `dosel_pct_media_periodo` y `urbano_pct_media_periodo` son resúmenes descriptivos del periodo. Se revisan sus valores y faltantes. **No** se deben confundir con los productos originales de Copernicus ni asumir equivalencia entre dosel y toda la vegetación.

In [ ]:
# 9. Resúmenes y gráficos de coberturas.
cob_col = ['dosel_pct_inicio_periodo','urbano_pct_inicio_periodo',
           'dosel_pct_media_periodo','urbano_pct_media_periodo']
cob_col = [c for c in cob_col if c in df]
resumen_cob = df.groupby('periodo',sort=False)[cob_col].agg(['count','median','mean','min','max'])
display(resumen_cob.round(3))
resumen_cob.to_csv(CARPETA/'resumen_coberturas.csv',encoding='utf-8-sig')
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for i,c in enumerate(['dosel_pct_media_periodo','urbano_pct_media_periodo']):
    ax[i].boxplot([pd.to_numeric(df.loc[df['periodo']==t,c],errors='coerce').dropna().values for t in df['periodo'].unique()], labels=list(df['periodo'].unique()), showfliers=False)
    ax[i].set(title=c.replace('_',' '),xlabel='',ylabel='Porcentaje (%)')
    ax[i].tick_params(axis='x',rotation=20)
plt.tight_layout();plt.savefig(CARPETA/'fig_coberturas_por_brote.png',dpi=170);plt.show()

## 4. Definición de los grupos de información
Se utilizan bloques metodológicamente interpretables. En la primera versión, `ia`, `ice`, `im`, `irs` se etiquetaron como «entomológicos», pero el documento de procedencia recibido los identifica como **índices territoriales y sociales**. Aquí se corrige esa descripción sin alterar sus valores.

- **Bioclimático:** `bio1` a `bio19`.
- **Índices territoriales/sociales:** `ia`, `ice`, `im`, `irs`.
- **Contexto humano y socioambiental:** `hdi`, población y huella humana disponible.
- **Coberturas históricas:** porcentaje de dosel y superficie construida, calculados con el procedimiento de Hansen/GHSL.

Los bloques son grupos de variables: varias variables pueden provenir de un mismo proveedor; «fuente» se usa aquí en sentido práctico. No se utiliza la etiqueta «entomológico» para índices que no miden directamente vectores.

In [ ]:
# 10. Construir los diccionarios de bloques de predictores.
BIO = [f'bio{i}' for i in range(1,20)]
GRUPOS_ANTERIORES = {
    'Clima bioclimático': BIO,
    'Índices territoriales/sociales': ['ia','ice','im','irs'],
    'Contexto socioambiental': ['hdi','poblacion_media_disponible','hfp_media_disponible']
}
GRUPOS_ACTUALIZADOS = {
    **GRUPOS_ANTERIORES,
    'Cobertura histórica Hansen/GHSL': ['dosel_pct_media_periodo','urbano_pct_media_periodo']
}
# Las covariables del 'primer año' no son necesariamente previas al brote.
# Se consideran una alternativa descriptiva/retrospectiva, NO pronóstico ex ante.
GRUPOS_INICIO = {
    'Clima bioclimático': BIO,
    'Índices territoriales/sociales': ['ia','ice','im','irs'],
    'Contexto al inicio': ['hdi','poblacion_inicio_periodo','hfp_inicio_periodo'],
    'Cobertura primer año': ['dosel_pct_inicio_periodo','urbano_pct_inicio_periodo']
}
for nombre, grupos in [('anterior',GRUPOS_ANTERIORES),('actualizado',GRUPOS_ACTUALIZADOS),('primer año',GRUPOS_INICIO)]:
    faltan = {k:[c for c in v if c not in df.columns] for k,v in grupos.items()}
    faltan = {k:v for k,v in faltan.items() if v}
    if faltan: raise ValueError(f'Faltan variables para el enfoque {nombre}: {faltan}')
    print(nombre.upper(), {k:len(v) for k,v in grupos.items()})

## 5. Diseño de validación temporal y prevención de fuga de información
- **Entrenamiento:** brotes 1 y 2 (2008–2009 y 2012–2013).
- **Validación exploratoria:** brote 3 (2018–2020). Se usa para comparar enfoques, **no es un conjunto final independiente** una vez que se consultan resultados para tomar decisiones.
- **Prueba reservada:** brote 4 (2023–2025). No se evalúa en este cuaderno para conservarlo como conjunto de prueba final.

Los promedios de cobertura de un periodo incluyen información posterior a su inicio. **El experimento actualizado con esos promedios tiene interpretación retrospectiva**, no permite afirmar capacidad de pronóstico antes del brote. Para predicción prospectiva deberán obtenerse covariables anteriores al periodo objetivo y considerar la duración y población expuesta.

Los modelos tienen pérdida Poisson para conteos y reciben imputaciones calculadas **solo en el entrenamiento**. La misma partición de validación se usa para todos los modelos.

In [ ]:
# 11. Construir las tres particiones sin mezclar tiempos.
train = df[df['brote'].isin([1,2])].copy()
valid = df[df['brote']==3].copy()
reservado = df[df['brote']==4].copy()
y_train = pd.to_numeric(train['casos_dengue'],errors='coerce')
y_valid = pd.to_numeric(valid['casos_dengue'],errors='coerce')
assert y_train.notna().all() and y_valid.notna().all()
assert (y_train>=0).all() and (y_valid>=0).all()
assert len(train)>0 and len(valid)>0 and len(reservado)>0
print('Entrenamiento:',train.shape,'Validación:',valid.shape,'Prueba reservada:',reservado.shape)
print('Media de casos (train):',y_train.mean().round(4),'(valid):',y_valid.mean().round(4))
print('El cuarto brote NO se utiliza para entrenar, ajustar ni evaluar en esta etapa.')

### 5.1. Funciones de imputación, entrenamiento y evaluación
Se aplica la mediana por columna calculada exclusivamente sobre el entrenamiento. Si una columna está completamente vacía en el entrenamiento, se utiliza cero **solo como marcador técnico dentro del modelo**, junto con una indicadora de ausencia, nunca como dato medido. Esto permite ejecutar la comparación pero se documenta como limitación. Los árboles pueden manejar faltantes, pero una política de preparación común hace comparables los bloques.

El modelo es `HistGradientBoostingRegressor` con pérdida Poisson, 85 iteraciones, tasa de aprendizaje de 0.065, 17 hojas máximas, mínimo de 35 observaciones por hoja y regularización L2 igual a 1.0. Son parámetros del experimento exploratorio previo, **no hiperparámetros optimizados**.

In [ ]:
# 12. Funciones reutilizables, comentadas paso a paso.
PARAMETROS = dict(loss='poisson',max_iter=85,learning_rate=0.065,
                  max_leaf_nodes=17,min_samples_leaf=35,
                  l2_regularization=1.0,early_stopping=False,random_state=SEMILLA)

def preparar_matrices(columnas):
    """Preparar X_train y X_valid sin consultar la distribución de validación para imputar."""
    # Convertir a numérico, eliminando infinitos.
    a = train[columnas].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    b = valid[columnas].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    # Estadísticos estimados SOLAMENTE con datos de entrenamiento.
    medianas = a.median(axis=0)
    sin_datos_train = medianas[medianas.isna()].index.tolist()
    # Añadir indicador cuando una variable no existe en toda la etapa de entrenamiento.
    if sin_datos_train:
        print('Aviso: columnas enteramente vacías en entrenamiento:', sin_datos_train)
    medianas = medianas.fillna(0.0)
    # Reemplazo por medianas del entrenamiento.
    return a.fillna(medianas), b.fillna(medianas), sin_datos_train

def predecir_poisson(cols):
    x_tr,x_va,vacias = preparar_matrices(cols)
    modelo = HistGradientBoostingRegressor(**PARAMETROS)
    modelo.fit(x_tr, y_train)
    return np.maximum(modelo.predict(x_va),0), modelo, vacias

def calcular_metricas(estrategia, pred, experimento):
    """MAE, RMSE y sesgo promedio de la predicción menos el valor observado."""
    pred=np.asarray(pred, dtype=float)
    return {'experimento':experimento,'estrategia':estrategia, 'validacion':'2018-2020',
            'n_validacion':len(y_valid),
            'MAE':mean_absolute_error(y_valid,pred),
            'RMSE':np.sqrt(mean_squared_error(y_valid,pred)),
            'sesgo_media':float(np.mean(pred-y_valid))}

def ejecutar_fusion(grupos, etiqueta):
    """Ajustar modelos individuales, fusión temprana y promedio de fusión tardía."""
    filas=[];preds={};modelos={};sin_datos={}
    # Un modelo por grupo: referentes unifuente para la fusión tardía.
    for nombre,cols in grupos.items():
        pred, modelo, vacias=predecir_poisson(cols)
        preds[nombre]=pred;modelos[nombre]=modelo;sin_datos[nombre]=vacias
        filas.append(calcular_metricas('Fuente individual: '+nombre,pred,etiqueta))
    # Fusión temprana: entrenar un modelo con TODAS las columnas (sin duplicados).
    todas=list(dict.fromkeys(c for cols in grupos.values() for c in cols))
    pred,modelo,vacias=predecir_poisson(todas)
    preds['Fusión temprana']=pred;modelos['Fusión temprana']=modelo
    sin_datos['Fusión temprana']=vacias
    filas.append(calcular_metricas('Fusión temprana',pred,etiqueta))
    # Fusión tardía: promedio NO ponderado de las predicciones individuales.
    pred_tardia=np.mean(np.column_stack([preds[n] for n in grupos]), axis=1)
    preds['Fusión tardía']=pred_tardia
    filas.append(calcular_metricas('Fusión tardía (media simple)',pred_tardia,etiqueta))
    return pd.DataFrame(filas).sort_values('RMSE'),preds,modelos,sin_datos

# Línea base constante, estimada SOLO con entrenamiento.
pred_ref=np.full(len(valid),float(y_train.mean()))
metrica_ref=calcular_metricas('Referencia: media histórica',pred_ref,'Referencia')
print('Referencia histórica:');display(pd.DataFrame([metrica_ref]))

## 6. Experimento 1: reproducción conceptual del primer avance (tres bloques)
Se reproduce el **procedimiento previo** con 19 variables bioclimáticas, los cuatro índices territoriales y el contexto socioambiental. Las coberturas nuevas **no se usan en este experimento**. La cifra exacta puede variar respecto al CSV histórico porque la base maestra y sus correcciones fueron actualizadas después de generar aquel resultado.

In [ ]:
# 13. Repetición de la comparación de tres bloques.
if EJECUTAR_MODELOS:
    res_anterior,pred_anterior,modelos_anterior,avisos_anterior = ejecutar_fusion(
        GRUPOS_ANTERIORES,'Tres bloques sin coberturas nuevas')
    res_anterior=pd.concat([res_anterior,pd.DataFrame([{**metrica_ref,
                                     'experimento':'Tres bloques sin coberturas nuevas'}])],ignore_index=True)
    res_anterior=res_anterior.sort_values('RMSE')
    display(res_anterior)
    res_anterior.to_csv(CARPETA/'metricas_experimento_1_tres_bloques.csv',index=False,encoding='utf-8-sig')
else:
    print('Modelos desactivados; configurar EJECUTAR_MODELOS=True para ejecutarlos.')

### 6.1. Contraste con el resultado histórico documentado
La siguiente tabla corresponde al **primer avance registrado anteriormente**, no a una ejecución de esta celda. Se conserva por trazabilidad. En aquel reporte el bloque `ia/ice/im/irs` figuraba con etiqueta «entomológico»; aquí se muestra como **índices territoriales/sociales**. Los resultados históricos no deben sustituirse silenciosamente por otros nuevos.

In [ ]:
# 14. Valores del primer experimento reportado anteriormente: SOLO como referencia documental.
# No se recalculan ni se hacen pasar por una evaluación independiente.
historico=pd.DataFrame([
 ('Referencia: media histórica',1.034267700703166,1.530301225004848,0.4027151969102931),
 ('Fuente individual: Contexto socioambiental',1.0323032433085626,1.5905458793007836,0.49251767665351015),
 ('Fusión tardía (media simple, 3 grupos)',1.0170161531998005,1.5999451488519405,0.4326627616679298),
 ('Fuente individual: Índices territoriales/sociales (etiqueta histórica corregida)',1.0427584181207552,1.687482504459316,0.4039189296121509),
 ('Fuente individual: Clima bioclimático',1.034551428681951,1.7975570895962765,0.4015516787381282),
 ('Fusión temprana (3 grupos)',1.102882556319013,2.0687936055198923,0.5163397981662267),
],columns=['estrategia','MAE','RMSE','sesgo_media'])
historico['n_validacion']=17089
historico['validacion']='2018-2020'
display(historico)
historico.to_csv(CARPETA/'metricas_historicas_primer_avance_NO_RECALCULADAS.csv',index=False,encoding='utf-8-sig')

## 7. Experimento 2: nueva fusión con las coberturas históricas (cuatro bloques)
Esta etapa **aún no formaba parte de los resultados del primer avance**. Aquí se entrenan de nuevo los modelos incorporando el bloque Hansen/GHSL, se usa la misma partición temporal y se compara con la referencia.

**Advertencia metodológica:** se incluyen promedios de dosel y superficie construida calculados dentro de cada periodo. Por tanto, este experimento representa una **predicción retrospectiva o asociación predictiva del periodo completo**, no un pronóstico previo a su inicio. No debe describirse como pronóstico ex ante.

In [ ]:
# 15. Entrenar y comparar la fusión con coberturas incluidas.
if EJECUTAR_MODELOS:
    res_actual,pred_actual,modelos_actual,avisos_actual = ejecutar_fusion(
        GRUPOS_ACTUALIZADOS,'Cuatro bloques con coberturas')
    res_actual=pd.concat([res_actual,pd.DataFrame([{**metrica_ref,
                                    'experimento':'Cuatro bloques con coberturas'}])],ignore_index=True)
    res_actual=res_actual.sort_values('RMSE')
    display(res_actual)
    res_actual.to_csv(CARPETA/'metricas_experimento_2_coberturas.csv',index=False,encoding='utf-8-sig')
    tabla_pred=valid[['brote','periodo','ciudad','id_fila_original','casos_dengue']].copy()
    tabla_pred['pred_referencia']=pred_ref
    for nombre,pred in pred_actual.items():
        clave=re.sub(r'[^a-z0-9_]+','_',unicodedata.normalize('NFKD',nombre.lower())
                     .encode('ascii','ignore').decode('ascii')).strip('_')
        tabla_pred['pred_'+clave]=pred
    tabla_pred.to_csv(CARPETA/'predicciones_validacion_experimento_2.csv',index=False,encoding='utf-8-sig')
else:
    print('Modelos desactivados; configurar EJECUTAR_MODELOS=True.')

## 8. Comparación de experimentos y visualización de métricas
Los resultados antiguos y nuevos se comparan **solo bajo la misma partición temporal y con las mismas métricas**. Si un modelo no mejora la referencia, se reporta sin modificar la conclusión. No se seleccionan pesos de fusión usando el cuarto brote.

In [ ]:
# 16. Comparativa visual con el experimento nuevo.
if EJECUTAR_MODELOS:
    comparacion=pd.concat([res_anterior,res_actual],ignore_index=True)
    display(comparacion[['experimento','estrategia','MAE','RMSE','sesgo_media']])
    comparacion.to_csv(CARPETA/'comparacion_experimentos.csv',index=False,encoding='utf-8-sig')
    fig,ax=plt.subplots(figsize=(12,6))
    datos=res_actual.sort_values('RMSE',ascending=False)
    ax.barh(datos['estrategia'],datos['RMSE'])
    ax.set(xlabel='RMSE (menor es mejor)',ylabel='',title='Validación temporal: fusión con coberturas históricas')
    plt.tight_layout();plt.savefig(CARPETA/'fig_rmse_modelos_actualizados.png',dpi=170);plt.show()
    print('Mejor RMSE del experimento nuevo:')
    display(res_actual.nsmallest(1,'RMSE'))

## 9. Experimento de sensibilidad: variables disponibles en el primer año
Este bloque contrasta el uso de promedios del periodo frente a las covariables del **primer año**. **No resuelve por sí mismo la fuga temporal:** un valor medido a mitad o fin del primer año tampoco estaba disponible al comienzo. Se presenta como análisis de sensibilidad retrospectivo.

Si se busca predicción anticipada real, se deberán reemplazar las variables por mediciones **previas** a cada periodo y determinar explícitamente el momento de pronóstico.

In [ ]:
# 17. Sensibilidad con variables del primer año.
if EJECUTAR_MODELOS:
    res_inicio,pred_inicio,modelos_inicio,avisos_inicio=ejecutar_fusion(
        GRUPOS_INICIO,'Variables del primer año (retrospectivo)')
    display(res_inicio)
    res_inicio.to_csv(CARPETA/'metricas_sensibilidad_primer_anio.csv',index=False,encoding='utf-8-sig')

## 10. Revisión cualitativa de errores (sin tocar el brote 4)
Se inspeccionan los errores de las observaciones de validación del tercer brote. Se trata de una ayuda para identificar problemas de modelado, no de una justificación para eliminar registros difíciles.

In [ ]:
# 18. Diagnóstico de residuales de la fusión temprana y tardía.
if EJECUTAR_MODELOS:
    err=pd.DataFrame({'observado':y_valid.to_numpy(),
                      'temprana':pred_actual['Fusión temprana'],
                      'tardia':pred_actual['Fusión tardía']})
    err['residuo_temprana']=err['observado']-err['temprana']
    err['residuo_tardia']=err['observado']-err['tardia']
    fig,ax=plt.subplots(1,2,figsize=(12,4.5))
    for eje,(nombre,r) in zip(ax,[('Fusión temprana','residuo_temprana'),('Fusión tardía','residuo_tardia')]):
        eje.scatter(err['observado'],err[r],alpha=.16,s=8)
        eje.axhline(0,linestyle='--',linewidth=1)
        eje.set(xlabel='Casos observados',ylabel='Observado - predicho',title=nombre)
    plt.tight_layout();plt.savefig(CARPETA/'fig_residuos_fusion.png',dpi=170);plt.show()
    err.to_csv(CARPETA/'errores_validacion.csv',index=False,encoding='utf-8-sig')

## 11. Exportación final y registro de decisiones
El ZIP contiene las tablas, métricas, predicciones y figuras generadas en las secciones anteriores. El cuaderno también guarda una bitácora breve de las decisiones metodológicas y limitaciones.

**Pendientes para la tesis:** (1) confirmar claves oficiales de AGEB y correspondencia geométrica; (2) resolver los polígonos sin datos de cobertura; (3) verificar que las variables y porcentajes se construyeron con la misma definición a lo largo del tiempo; (4) definir población-tiempo de exposición para comparar brotes de duración distinta; (5) elaborar un escenario de predicción realmente anticipada; (6) realizar ajuste y selección solo con entrenamiento/validación; (7) preservar el cuarto brote hasta la prueba final; (8) considerar validación espacial por ciudad.

**Interpretación:** RMSE y MAE menores implican menor error. Un sesgo positivo (`predicción − observado`) indica sobreestimación promedio. Los resultados son preliminares hasta completar las comprobaciones señaladas.

In [ ]:
# 19. Archivo de bitácora y ZIP para descargar.
bitacora = {
  'fecha_cuaderno':'2026-11-05',
  'base_maestra':str(archivo),
  'filas':int(len(df)),'columnas':int(len(df.columns)),
  'periodos':sorted(df['periodo'].dropna().unique().tolist()),
  'entrenamiento':['2008-2009','2012-2013'],
  'validacion_exploratoria':'2018-2020',
  'conjunto_reservado':'2023-2025',
  'metodo_fusion_tardia':'promedio simple de predicciones por bloque',
  'metodo_fusion_temprana':'modelo con todas las variables de los bloques',
  'modelo':'HistGradientBoostingRegressor(loss=poisson)',
  'semilla':SEMILLA,
  'interpretacion':'retrospectiva cuando se usan promedios dentro del brote',
  'limitaciones':['revisión geométrica pendiente',
                  'faltantes por descargas incompletas',
                  'duración de brotes diferente',
                  'no evaluar el cuarto brote hasta prueba final',
                  'índices ia/ice/im/irs no son entomológicos'],
}
with open(CARPETA/'bitacora_metodologica.json','w',encoding='utf-8') as f:
    json.dump(bitacora,f,ensure_ascii=False,indent=2)
zip_final=Path('Reportes_Completos_Fusion_Multifuente.zip')
with zipfile.ZipFile(zip_final,'w',zipfile.ZIP_DEFLATED) as z:
    for archivo_salida in CARPETA.iterdir():
        if archivo_salida.is_file():
            z.write(archivo_salida,arcname=archivo_salida.name)
print('Archivos generados:',len([p for p in CARPETA.iterdir() if p.is_file()]))
print('ZIP de reportes:',zip_final.resolve())
try:
    from google.colab import files
    files.download(str(zip_final))
except ImportError:
    print('Fuera de Colab, el ZIP está guardado en el directorio actual.')

## Referencias metodológicas para documentar en la tesis
- Breiman, L. (2001). Random forests. *Machine Learning, 45*, 5–32. https://doi.org/10.1023/A:1010933404324
- Fick, S. E., & Hijmans, R. J. (2017). WorldClim 2: New 1-km spatial resolution climate surfaces for global land areas. *International Journal of Climatology, 37*(12), 4302–4315. https://doi.org/10.1002/joc.5086
- Hansen, M. C., et al. (2013). High-resolution global maps of 21st-century forest cover change. *Science, 342*(6160), 850–853. https://doi.org/10.1126/science.1244693
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html
- European Commission, Joint Research Centre. (2023). *GHSL global built-up surface 1975–2030 (GHS-BUILT-S R2023A)*. https://human-settlement.emergency.copernicus.eu/
- Google Earth Engine. (s. f.). *Copernicus Global Land Cover Layers: CGLS-LC100 collection 3*. https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_Landcover_100m_Proba-V-C3_Global

Las referencias sirven para describir los productos y las herramientas. La definición específica de cada variable debe contrastarse con los metadatos del archivo original; las fuentes disponibles no sustituyen la verificación de la generación de los polígonos.